# Three-method DeePC comparison

Independently tuned **Select-DPC**, **Contextual DPC**, and the current **Atlas-DeePC**.
Atlas uses **uncentered linear rank-60 charts and iterative N=10 terminal-reference generation**.
The two external methods optimize raw trajectory coefficients; they do not receive Atlas SVD bases or graph references.

This notebook is separate from the historical, running `comparison.ipynb`. Read
[the comparison contract](../../docs/THREE_METHOD_COMPARISON.md) for source versions and Reacher adaptations.
The contextual default implements the paper's Eq. (4); its penalty norms differ from the latest public code.


## 1. Setup and protocol

Use the `gym-robotics-mujoco` kernel. Expensive stages are disabled below.
The result directory has its own plan, checkpoints, and source/data fingerprints.
Before the first run, edit the plan if needed. After a run starts, use a new directory to change the protocol.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
from IPython.display import Image, display
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "baseline_atlas_deepc.py").is_file())
assert (ROOT / "src/three_method_benchmark.py").exists(), "Open from the project root."
sys.path.insert(0, str(ROOT / "src"))
import three_method_benchmark as benchmark
OUTPUT_DIR = ROOT / "experiments/three_method_faithful_v1"

def run_phase(phase):
    subprocess.run(
        [sys.executable, "src/three_method_benchmark.py", phase, "--output-dir", str(OUTPUT_DIR)],
        cwd=ROOT, check=True,
    )

if not (OUTPUT_DIR / "plan.json").exists():
    run_phase("init")
plan = json.loads((OUTPUT_DIR / "plan.json").read_text())
benchmark.validate_plan(plan)
display(pd.DataFrame([
    {"method": "Select-DPC", "representation": "Raw columns, sum(g)=1",
     "reference": "Final fingertip target", "online work": "L1 selection + five QP refinements"},
    {"method": "Contextual DPC", "representation": "Normalized raw columns, sum(g)=1",
     "reference": "Final fingertip target", "online work": "Contextual score + one solve"},
    {"method": "Atlas-DeePC", "representation": "48 uncentered linear charts, rank 60, zero offset",
     "reference": "Graph routing + iterative N=10 terminal selection",
     "online work": "Chart lookup + candidate selection/routing + one QP"},
]).set_index("method"))
print("Contextual objective:", plan["methods"]["contextual_dpc"]["base"]["formulation"])
print("Plan:", OUTPUT_DIR / "plan.json")
print("Development:", plan["development_seeds"][0], "through", plan["development_seeds"][-1])
print("Reserved test:", plan["test_seeds"][0], "through", plan["test_seeds"][-1])


## 2. Review the grid-search budget

Each method gets two stages of 27 combinations by default. Each stage screens on 8 shared development seeds and evaluates five finalists on all 32 development seeds. The incumbent occupies one finalist slot if needed. The second stage starts from the first stage's winner. This is a finite, staged search; it does not guarantee a global optimum.

Rank candidates by sustained success, then by mean fixed task cost over all development seeds. The same configuration is used for every final test seed. Atlas's 48 charts, rank 60, and N=10 policy remain fixed.


In [ ]:
grid_rows = []
for method, spec in plan["methods"].items():
    for stage_index, grid in enumerate(spec["stages"], 1):
        combinations = 1
        for values in grid.values():
            combinations *= len(values)
        budget = combinations * plan["screening_seeds"] + plan["finalists"] * (
            len(plan["development_seeds"]) - plan["screening_seeds"]
        )
        grid_rows.append({
            "method": benchmark.LABELS[method], "stage": stage_index,
            "grid": grid, "grid combinations": combinations,
            "rollout budget before cache reuse": budget,
        })
display(pd.DataFrame(grid_rows))
print("Raw Select-DPC runs five refinements per step. The full study can take days, not hours.")
print("Review the smoke timings and grid budget before enabling tuning or the final test.")
print("The final test has", len(plan["test_seeds"]) * 3, "rollouts.")


## 3. Smoke test

This checks short rollouts on a separate seed. It is not a success-rate experiment and does not tune parameters. All controllers have the same physical initialization and torque limits. Run this after the other heavy experiment finishes if you want useful timing estimates.


In [ ]:
RUN_SMOKE = False
if RUN_SMOKE:
    run_phase("smoke")
path = OUTPUT_DIR / "smoke_summary.json"
if path.exists():
    smoke = json.loads(path.read_text())
    display(pd.DataFrame(smoke["rows"])[[
        "method", "seed", "steps", "unsolved_subproblems",
        "input_bound_violations", "median_compute_ms", "p99_compute_ms", "max_compute_ms",
    ]])
    if any(row["unsolved_subproblems"] for row in smoke["rows"]):
        print("WARNING: smoke test has unsolved optimizations. Inspect numerical reliability on development/smoke seeds before a long run.")
    estimate = pd.DataFrame([{
        "method": row["method"],
        "rough test hours at smoke median": row["median_compute_ms"] / 1000 * plan["steps"] * len(plan["test_seeds"]) / 3600,
    } for row in smoke["rows"]])
    display(estimate)
    print("Rough planning estimate only: excludes setup/tuning; selected weights, horizons and solver failures can change runtime substantially.")
else:
    print("Set RUN_SMOKE=True to run short validation rollouts.")


## 4. Tune on development seeds

This stage is resumable per configuration and seed. The test panel is not evaluated. It preserves each method's penalty types and representation while searching that method's own parameter grid.


In [ ]:
RUN_TUNING = False
if RUN_TUNING:
    run_phase("tune")
progress = OUTPUT_DIR / "tuning_progress.json"
if progress.exists():
    history = json.loads(progress.read_text())
    display(pd.DataFrame([
        {"method": benchmark.LABELS[method], "completed stage": stage["stage"], "winner": stage["winner"]}
        for method, stages in history.items() for stage in stages
    ]))
else:
    print("No completed tuning stage yet.")


## 5. Review and freeze one configuration per method

Review the development winners before enabling freeze. Freezing records the selected configurations, code/data fingerprints, fixed evaluation cost, and reserved test seeds. Tuning in that directory is then closed.


In [ ]:
tuning_path = OUTPUT_DIR / "tuning.json"
if tuning_path.exists():
    tuning = json.loads(tuning_path.read_text())
    display(pd.DataFrame(tuning["selected"]).T)
else:
    print("Complete tuning before freezing.")
FREEZE_SELECTION = False
if FREEZE_SELECTION:
    run_phase("freeze")
print("Frozen:", (OUTPUT_DIR / "frozen.json").exists())


## 6. Final paired test

One frozen configuration per method, on the reserved test panel. Serial runs rotate method order across seeds and use one numerical thread. Each rollout is saved immediately and can be resumed. For timing claims, run without competing heavy workloads.


In [ ]:
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    run_phase("test")
else:
    print("Final test disabled. Complete tuning and freeze, then set RUN_FINAL_TEST=True.")


## 7. Display results and regenerate plots — no controller reruns

Run only this cell whenever you want refreshed tables or figures. Partial results include only seeds completed by all three methods.

The **common task cost** always uses the requested fingertip target and fixed evaluation weights, including input effort and input changes. Tuning weights and changing Atlas waypoints do not change this metric. Median/p99 timing pool steady control steps across all paired seeds; max includes initialization transients. These quantities are not medians of rollout p99s.


In [ ]:
run_phase("report")  # Reads saved results only; never launches rollouts.
summary_path = OUTPUT_DIR / "summary.json"
if summary_path.exists():
    result = json.loads(summary_path.read_text())
    print("FINAL" if result["complete"] else "PARTIAL", "-", len(result["paired_seeds"]), "paired seeds")
    table = pd.DataFrame(result["methods"]).set_index("label")
    display(table[[
        "successes", "total", "success_rate", "wilson_95", "median_final100_mm",
        "median_common_cost", "mean_common_cost", "median_compute_ms",
        "p99_compute_ms", "max_compute_ms", "unsolved_subproblems", "input_bound_violations",
    ]])
    display(Image(data=(OUTPUT_DIR / "three_method_comparison.png").read_bytes()))
    display(pd.DataFrame(result["paired_comparisons"]))
else:
    print("No complete three-method test triplets yet.")


## Interpretation and provenance

This study compares independently tuned Reacher adaptations of published methods against the current linear/iterative Atlas baseline. It does not reuse the historical SVD-modified selectors as native baselines.

Select retains its original raw QP and refinement loop. Contextual retains raw trajectory coefficients, training-only normalization, its contextual score, and the selected paper/code objective version. The Reacher port uses actual torque inputs; the vehicle/drone-specific spatial alignment and incremental-input augmentation are not included. This adaptation and the paper/code discrepancy must be disclosed when writing the paper.

The search and hold-out split support a method-level comparison. The previous notebook remains useful for component ablations. Neither a short smoke test nor a development-grid winner is a final performance claim.
